<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/14_NLP_Applications/14_01_NLP_Applications.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14_01 NLP 응용 사례 : 정보 추출 · 질의응답 · 요약 · 대화

**학습 목표**
- **정보 추출(NER)**, **질의응답(추출형 QA)**, **문서 요약(추상 요약)**, **대화 시스템**을 직접 실행해 본다.
- Hugging Face `transformers` **파이프라인(pipeline)** 으로 대표 NLP 응용을 체험한다.

> ※ 사전학습 모델을 내려받아 실행하므로 첫 실행에 시간이 걸립니다(Colab GPU 권장). 모델을 못 찾는 경우 주석의 대체 모델로 바꿔 보세요.

In [ ]:
!pip install -q transformers torch sentencepiece

## 1. 정보 추출 — 개체명 인식(NER)

비정형 문장에서 **인물·장소·기관** 등 개체를 찾아 유형을 분류합니다.

In [ ]:
from transformers import pipeline

# 한국어 NER 모델 (없으면 다른 한국어 NER 모델로 교체 가능)
try:
    ner = pipeline('token-classification',
                   model='Leo97/KoELECTRA-small-v3-modu-ner',
                   aggregation_strategy='simple')
    text = '홍길동은 2017년에 서울에서 네이버에 입사했다.'
    for ent in ner(text):
        print(f"{ent['word']:10s} -> {ent['entity_group']}  (score={ent['score']:.2f})")
except Exception as e:
    print('NER 모델 로드 실패:', e)
    print('→ 다른 한국어 NER 모델(예: KPF/KPF-bert-ner)로 교체해 보세요.')

## 2. 질의응답(QA) — 추출형(Extractive)

**지문(context)** 에서 질문의 정답 **구간(span)** 을 그대로 찾아냅니다. 한국어 표준 벤치마크는 **KorQuAD** 입니다.

In [ ]:
qa = pipeline('question-answering',
              model='monologg/koelectra-base-v3-finetuned-korquad')

context = (
    '트랜스포머는 2017년 구글이 발표한 셀프 어텐션 기반 모델이다. '
    'BERT는 트랜스포머의 인코더를 사용해 2018년에 공개되었다.'
)
for q in ['트랜스포머는 언제 발표되었나?', 'BERT는 무엇을 사용하는가?']:
    ans = qa(question=q, context=context)
    print(f"Q: {q}\nA: {ans['answer']}  (score={ans['score']:.2f})\n")

## 3. 문서 요약 — 추상 요약(Abstractive)

원문을 **이해해 새 문장으로 재작성**합니다. 여기서는 한국어 **KoBART** 요약 모델을 사용합니다.

In [ ]:
summarizer = pipeline('summarization', model='gogamza/kobart-summarization')

doc = (
    '대규모 언어모델(LLM)의 등장으로 자연어처리 응용이 크게 확대되었다. '
    '과거에는 태스크마다 별도의 모델을 학습해야 했지만, 이제는 하나의 범용 모델에 '
    '프롬프트로 지시하여 번역, 요약, 질의응답, 코드 생성 등 다양한 작업을 수행할 수 있다. '
    '다만 환각과 평가, 윤리 같은 새로운 과제도 함께 등장했다.'
)
summary = summarizer(doc, max_length=60, min_length=15)[0]['summary_text']
print('요약:', summary)

## 4. 대화 시스템 — 태스크 지향형(Task-oriented) 미니 챗봇

특정 목적(예: 주문)을 달성하기 위해 **의도 파악**과 **슬롯 채우기(Slot Filling)** 를 수행합니다. (개방형 대화는 12-3에서 다룬 LLM API로 구현할 수 있습니다.)

In [ ]:
MENU = ["페퍼로니", "치즈", "불고기"]
SIZES = ["스몰", "미디엄", "라지"]

def make_bot():
    slots = {"메뉴": None, "크기": None}
    def respond(utterance):
        for m in MENU:
            if m in utterance:
                slots["메뉴"] = m
        for s in SIZES:
            if s in utterance:
                slots["크기"] = s
        missing = [k for k, v in slots.items() if v is None]
        if missing:
            return f"{', '.join(missing)}을(를) 알려주세요."
        return f"주문 완료: {slots['메뉴']} 피자 {slots['크기']} 사이즈"
    return respond

bot = make_bot()
for u in ["불고기 피자 주문할게요", "라지로 주세요"]:
    print("사용자:", u)
    print("봇:", bot(u), "\n")

## 5. 정리

| 응용 | 사용한 도구 |
|------|-------------|
| 정보 추출(NER) | `pipeline('token-classification')` |
| 질의응답(추출형 QA) | `pipeline('question-answering')` · KorQuAD |
| 문서 요약(추상) | `pipeline('summarization')` · KoBART |
| 대화(태스크 지향형) | 의도 파악 + 슬롯 채우기 |

> 💡 LLM 시대에는 이 응용들을 **프롬프트/RAG**로 통합해 하나의 모델로 처리하는 경우가 많습니다(12~13주 참고).